# 3. Visualizing results

`scope-profiler` ships four chart types, all built on
[maxplotlib](https://github.com/max-models/maxplotlib):

| function | answers |
| --- | --- |
| `plot_gantt` | when did each call happen, on which rank? |
| `plot_flame` | how does time break down across nested regions? |
| `plot_durations` | which regions cost the most (total / avg / min / max)? |
| `plot_speedup` | how does a region scale with ranks or threads? |

They need the `pproc` extra:

```bash
pip install "scope-profiler[pproc]"
```

Every function takes a results (or a list of runs), `include` / `exclude`
filters, a `filepath` to save to, `show=True` to display, and a `backend` of
either `"matplotlib"` (default) or `"plotly"` (interactive).

In [ ]:
import tempfile
import time
from pathlib import Path

from scope_profiler import ProfileManager, read_h5

WORKDIR = Path(tempfile.mkdtemp(prefix="scope-profiler-tutorial-"))
DATA_FILE = WORKDIR / "profiling_data.h5"

ProfileManager.setup(file_path=str(DATA_FILE))

with ProfileManager.profile_region("setup"):
    time.sleep(0.02)

for step in range(4):
    with ProfileManager.profile_region("timestep"):
        with ProfileManager.profile_region("assemble"):
            time.sleep(0.006)
        with ProfileManager.profile_region("solve"):
            time.sleep(0.010)
        with ProfileManager.profile_region("output"):
            time.sleep(0.002)

ProfileManager.finalize(verbose=False)
results = read_h5(DATA_FILE)
results.print_summary()

In a notebook the simplest approach is to save each chart as a PNG and display
it. (PNG export with the default matplotlib backend needs nothing extra; the
plotly backend writes self-contained `.html`, or `.png` if `kaleido` is
installed.)

In [ ]:
from IPython.display import Image, display

from scope_profiler import plot_durations, plot_flame, plot_gantt

## Gantt chart — what ran when

Each call becomes a bar on a timeline, with one lane per rank. This is the chart
for spotting gaps, stragglers and serialization.

In [ ]:
gantt_path = WORKDIR / "gantt.png"
plot_gantt(results, filepath=str(gantt_path), verbose=False)
display(Image(str(gantt_path)))

## Flame chart — where the time goes

The flame chart reconstructs the nesting from the timestamps: a region drawn on
top of another ran inside it. It covers rank 0 by default, since it represents a
single execution's call stack; pass `ranks=[...]` for one chart per rank.

In [ ]:
flame_path = WORKDIR / "flame.png"
plot_flame(results, filepath=str(flame_path), verbose=False)
display(Image(str(flame_path)))

## Duration charts — which region is expensive

`plot_durations` draws one bar chart per metric. By default it produces all of
`total`, `avg`, `min` and `max`, writing one file per metric and returning the
paths it wrote; pass `metrics=` to narrow that down.

In [ ]:
durations_paths = plot_durations(
    results,
    metrics=["total", "avg"],
    filepath=str(WORKDIR / "durations.png"),
    verbose=False,
)
for path in durations_paths:
    print(path)
    display(Image(path))

## Filtering and colors

`include` / `exclude` are regexes matched against region names, and `cmap` picks
any [matplotlib colormap](https://matplotlib.org/stable/users/explain/colors/colormaps.html).
Region colors are consistent across chart types, so a region keeps its color
between the Gantt and flame views.

In [ ]:
filtered_path = WORKDIR / "gantt_filtered.png"
plot_gantt(
    results,
    include=["solve", "assemble"],
    cmap="viridis",
    filepath=str(filtered_path),
    verbose=False,
)
display(Image(str(filtered_path)))

## Speedup charts

`plot_speedup` compares the *same* region across several files and plots speedup
against a metadata field — `num_ranks` (default), `omp_num_threads` or
`total_cores`. It needs at least two files, so it is normally fed a scaling
study:

```python
runs = [read_h5(f"run_{n}ranks.h5") for n in (1, 2, 4, 8)]
plot_speedup(runs, x_field="num_ranks", filepath="speedup.png")
```

To keep this notebook serial, the cell below writes three small HDF5 files by
hand that imitate a 1/2/4-thread run — the shape of a real scaling study without
needing the cores.

In [ ]:
import h5py
import numpy as np

from scope_profiler import plot_speedup

scaling_files = []
for threads in (1, 2, 4):
    path = WORKDIR / f"scaling_{threads}.h5"
    duration_ns = int(0.4e9 / threads)  # perfect scaling, for illustration
    with h5py.File(path, "w") as handle:
        meta = handle.create_group("metadata")
        meta.attrs["omp_num_threads"] = threads
        meta.attrs["mpi_size"] = 1
        meta.attrs["total_cores"] = threads
        group = handle.create_group("rank0/regions/solve")
        group.create_dataset("start_times", data=np.array([0], dtype=np.int64))
        group.create_dataset("end_times", data=np.array([duration_ns], dtype=np.int64))
    scaling_files.append(path)

speedup_path = WORKDIR / "speedup.png"
plot_speedup(
    [read_h5(path) for path in scaling_files],
    x_field="omp_num_threads",
    filepath=str(speedup_path),
    verbose=False,
)
display(Image(str(speedup_path)))

## Exporting the plotted numbers

Every plotting function takes `data_filepath` (and `data_format` of `"csv"` or
`"json"`) to write out exactly the values it drew — handy when you want the
chart in one tool and the numbers in another.

In [ ]:
plot_durations(
    results,
    metrics="total",
    filepath=str(WORKDIR / "durations_total.png"),
    data_filepath=WORKDIR / "durations.csv",
    data_format="csv",
    verbose=False,
)
print((WORKDIR / "durations.csv").read_text())

## The same thing from the command line

Everything above is also one CLI invocation, which is usually what you want on a
cluster:

```bash
scope-profiler pproc profiling_data.h5 -o figures
scope-profiler pproc profiling_data.h5 -o figures --backend plotly --cmap viridis
scope-profiler pproc 'run_*.h5' -o figures --export data --export-data-format json
```

`pproc` accepts several files (or a glob) and writes the Gantt, flame, duration
and — for multiple files — speedup charts, plus `region_statistics.json`.

## Next

[4. Profiling modes](04_profiling_modes.ipynb) covers the configuration options
that decide what gets recorded in the first place.